In [1]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import os

In [2]:
'''
Данные
'''

import gc

X_train = pl.read_parquet('/home/f.danilov/workflow/kiber-polka/data/train/train_main_features.parquet')
y_train = pl.read_parquet('/home/f.danilov/workflow/kiber-polka/data/train/train_target.parquet')
meta_data = pl.read_parquet('/home/f.danilov/workflow/kiber-polka/meta_data.parquet')
X_train_pd = X_train.drop('customer_id').to_pandas()
y_train_pd = y_train.drop('customer_id').to_pandas()
meta_data_pd = meta_data.to_pandas()

del meta_data, y_train, X_train
gc.collect()

0

In [3]:
snapshots_path = r'/home/f.danilov/workflow/kiber-polka/snapshots'
snapshots = sorted(
    [os.path.join(snapshots_path, snapshot) for snapshot in os.listdir(snapshots_path)], key=lambda x: (len(x), x))
configs_path = r'/home/f.danilov/workflow/kiber-polka/configs'
configs   = sorted(
    [os.path.join(configs_path, config) for config in os.listdir(configs_path)], key=lambda x: (len(x), x))

In [4]:
targets = sorted([snapshot.replace('.cbm', '') for snapshot in os.listdir(snapshots_path)], key=lambda x: (len(x), x))
targets

['target_1_1',
 'target_1_2',
 'target_1_3',
 'target_1_4',
 'target_1_5',
 'target_2_1',
 'target_2_2',
 'target_2_3',
 'target_2_4',
 'target_2_5',
 'target_2_6',
 'target_2_7',
 'target_2_8',
 'target_3_1',
 'target_3_2',
 'target_3_3',
 'target_3_4',
 'target_3_5',
 'target_4_1',
 'target_5_1',
 'target_5_2',
 'target_6_1',
 'target_6_2',
 'target_6_3',
 'target_6_4',
 'target_6_5',
 'target_7_1',
 'target_7_2',
 'target_7_3',
 'target_8_1',
 'target_8_2',
 'target_8_3',
 'target_9_1',
 'target_9_2',
 'target_9_3',
 'target_9_4',
 'target_9_5',
 'target_9_6',
 'target_9_7',
 'target_9_8',
 'target_10_1']

In [5]:
'''
Мета-модели
'''

import joblib, json
from sklearn.ensemble import HistGradientBoostingClassifier

meta_models = {}
feature_order = {}

for i, target_idx in enumerate(targets):
    cbm_path    = snapshots[i]
    config_path = configs[i]

    print(f"\n[{i}/41] {target_idx}")

    with open(config_path, 'r', encoding='utf-8') as file:
        config = json.load(file)

    y_target = y_train_pd[target_idx].values

    features = config['best_features']

    available_features = [f for f in features if f in X_train_pd.columns]

    X_combined = pd.concat([
        X_train_pd[available_features].reset_index(drop=True),
        meta_data_pd[[target_idx]].reset_index(drop=True)
    ], axis=1)

    # Мета-модель
    meta_model = HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.03,
        random_state=42,
        verbose=0
    )

    feature_order[target_idx] = X_combined.columns.tolist()
    
    meta_model.fit(X_combined, y_target)

    joblib.dump(meta_model, f'meta_snapshots/{target_idx}.pkl')

    meta_models[target_idx] = meta_model
    
    # Оценка
    y_pred = meta_model.predict_proba(X_combined)[:, 1]
    auc = roc_auc_score(y_target, y_pred)
    
    print(f"  AUC: {auc:.4f}")

import pickle

with open('meta_model/feature_order.pkl', 'wb') as f:
    pickle.dump(feature_order, f)

with open('meta_model/meta_models.pkl', 'wb') as f:
    pickle.dump(meta_models, f)



[0/41] target_1_1
  AUC: 0.9355

[1/41] target_1_2
  AUC: 0.9141

[2/41] target_1_3
  AUC: 0.8907

[3/41] target_1_4
  AUC: 0.8561

[4/41] target_1_5
  AUC: 0.9690

[5/41] target_2_1
  AUC: 0.8925

[6/41] target_2_2
  AUC: 0.9295

[7/41] target_2_3
  AUC: 0.9460

[8/41] target_2_4
  AUC: 0.8394

[9/41] target_2_5
  AUC: 0.9251

[10/41] target_2_6
  AUC: 0.8719

[11/41] target_2_7
  AUC: 0.8124

[12/41] target_2_8
  AUC: 0.7881

[13/41] target_3_1
  AUC: 0.7006

[14/41] target_3_2
  AUC: 0.9080

[15/41] target_3_3
  AUC: 0.9120

[16/41] target_3_4
  AUC: 0.9755

[17/41] target_3_5
  AUC: 0.9906

[18/41] target_4_1
  AUC: 0.9049

[19/41] target_5_1
  AUC: 0.8047

[20/41] target_5_2
  AUC: 0.8456

[21/41] target_6_1
  AUC: 0.7960

[22/41] target_6_2
  AUC: 0.8267

[23/41] target_6_3
  AUC: 0.8723

[24/41] target_6_4
  AUC: 0.8818

[25/41] target_6_5
  AUC: 0.9224

[26/41] target_7_1
  AUC: 0.8136

[27/41] target_7_2
  AUC: 0.8784

[28/41] target_7_3
  AUC: 0.8937

[29/41] target_8_1
  AU

In [6]:
X_test = pl.read_parquet('/home/f.danilov/workflow/kiber-polka/data/test/test_main_features_typed.parquet')
idxs = X_test['customer_id']
X_test = X_test.drop('customer_id')

X_test_pd = X_test.to_pandas()
X_test_pd.shape

(250000, 200)

In [8]:

predictions = {}
predictions['customer_id'] = idxs
test_meta = {}

# Предсказания базовых моделей
for i, ind_target in enumerate(targets):
    cbm_path    = snapshots[i]
    config_path = configs[i]

    with open(config_path, 'r', encoding='utf-8') as file:
        config = json.load(file)

    model = CatBoostClassifier().load_model(cbm_path)
    required_features = model.feature_names_

    #available_features = [f for f in required_features if f in X_test_pd.columns]

    sub_X_test = X_test.select(required_features)

    X_test_pl = Pool(
        data=sub_X_test, 
        cat_features=[feature for feature in config['category'] if feature in required_features],
    )

    preds = model.predict_proba(X_test_pl)[:, 1]
    test_meta[ind_target] = preds

test_meta_df = pd.DataFrame(test_meta)
test_meta_df

,target_1_1,target_1_2,target_1_3,target_1_4,target_1_5,target_2_1,target_2_2,target_2_3,target_2_4,target_2_5,...,target_8_3,target_9_1,target_9_2,target_9_3,target_9_4,target_9_5,target_9_6,target_9_7,target_9_8,target_10_1
0,0.000638,0.001598,0.009127,0.012016,0.001051,0.023939,0.004379,0.000548,0.006821,0.001020,...,0.025349,0.003948,0.065696,0.020327,0.000144,0.002555,0.372243,0.120422,0.002068,0.324144
1,0.008744,0.002297,0.047046,0.019958,0.002395,0.034314,0.010797,0.000157,0.018941,0.001733,...,0.007344,0.001178,0.195957,0.025139,0.018067,0.018433,0.376068,0.108416,0.000222,0.286678
2,0.002539,0.004935,0.010710,0.036948,0.001305,0.007839,0.003274,0.000729,0.002488,0.001825,...,0.011547,0.004290,0.042120,0.012466,0.000070,0.000940,0.194164,0.061853,0.001928,0.336926
3,0.000136,0.000617,0.002451,0.005796,0.000126,0.008514,0.001632,0.000235,0.007614,0.000328,...,0.016858,0.003031,0.038164,0.021370,0.000149,0.003459,0.361655,0.135696,0.004437,0.321799
4,0.010492,0.002067,0.021243,0.022564,0.000325,0.003605,0.000811,0.000552,0.001496,0.002214,...,0.006530,0.002883,0.058727,0.025119,0.000238,0.008508,0.230532,0.112185,0.000673,0.319570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,0.002629,0.000775,0.002688,0.007107,0.000469,0.000237,0.001035,0.000080,0.003260,0.000553,...,0.002249,0.000230,0.001497,0.013136,0.003268,0.001435,0.111951,0.006380,0.000381,0.728206
249996,0.001605,0.001744,0.007503,0.006704,0.001076,0.012099,0.001434,0.019359,0.002839,0.000602,...,0.002953,0.001115,0.022340,0.007794,0.000283,0.000781,0.086579,0.049401,0.014530,0.272837
249997,0.000395,0.000133,0.000446,0.001135,0.000091,0.000471,0.000082,0.000616,0.004000,0.001179,...,0.003505,0.000569,0.002080,0.005776,0.000010,0.000161,0.014305,0.005652,0.000517,0.070203
249998,0.005858,0.002417,0.002908,0.012260,0.001628,0.009614,0.001884,0.000847,0.002835,0.000721,...,0.003321,0.004737,0.060475,0.011774,0.000148,0.003732,0.234432,0.098802,0.004414,0.273993


In [10]:
with open('meta_model/feature_order.pkl', 'rb') as f:
    feature_order = pickle.load(f)

# Предсказания мета-моделей
for i, target_idx in enumerate(targets):
    print(f"  [{i}/41] {target_idx}")
    config_path = configs[i]
    
    with open(config_path, 'r', encoding='utf-8') as file:
        config = json.load(file)

    features = config['best_features']
    available_features = [f for f in features if f in X_test_pd.columns]
    
    X_subset = X_test_pd[available_features]
    X_combined = pd.concat([
        X_subset.reset_index(drop=True),
        test_meta_df[[target_idx]].reset_index(drop=True)
    ], axis=1)

    expected_features = feature_order[target_idx]
    X_combined = X_combined[expected_features]
    
    preds = meta_models[target_idx].predict_proba(X_combined)[:, 1]
    
    predictions[config['target'].replace('target', 'predict', 1)] = preds


  [0/41] target_1_1
  [1/41] target_1_2
  [2/41] target_1_3
  [3/41] target_1_4
  [4/41] target_1_5
  [5/41] target_2_1
  [6/41] target_2_2
  [7/41] target_2_3
  [8/41] target_2_4
  [9/41] target_2_5
  [10/41] target_2_6
  [11/41] target_2_7
  [12/41] target_2_8
  [13/41] target_3_1
  [14/41] target_3_2
  [15/41] target_3_3
  [16/41] target_3_4
  [17/41] target_3_5
  [18/41] target_4_1
  [19/41] target_5_1
  [20/41] target_5_2
  [21/41] target_6_1
  [22/41] target_6_2
  [23/41] target_6_3
  [24/41] target_6_4
  [25/41] target_6_5
  [26/41] target_7_1
  [27/41] target_7_2
  [28/41] target_7_3
  [29/41] target_8_1
  [30/41] target_8_2
  [31/41] target_8_3
  [32/41] target_9_1
  [33/41] target_9_2
  [34/41] target_9_3
  [35/41] target_9_4
  [36/41] target_9_5
  [37/41] target_9_6
  [38/41] target_9_7
  [39/41] target_9_8
  [40/41] target_10_1


In [11]:
submit = pl.DataFrame(predictions)
submit.write_parquet('submit.parquet')

In [12]:
d = pl.read_parquet('submit.parquet')
d

customer_id,predict_1_1,predict_1_2,predict_1_3,predict_1_4,predict_1_5,predict_2_1,predict_2_2,predict_2_3,predict_2_4,predict_2_5,predict_2_6,predict_2_7,predict_2_8,predict_3_1,predict_3_2,predict_3_3,predict_3_4,predict_3_5,predict_4_1,predict_5_1,predict_5_2,predict_6_1,predict_6_2,predict_6_3,predict_6_4,predict_6_5,predict_7_1,predict_7_2,predict_7_3,predict_8_1,predict_8_2,predict_8_3,predict_9_1,predict_9_2,predict_9_3,predict_9_4,predict_9_5,predict_9_6,predict_9_7,predict_9_8,predict_10_1
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,0.000949,0.001846,0.00579,0.009024,0.001258,0.03466,0.009488,0.000931,0.008085,0.002188,0.002881,0.000257,0.000086,0.126875,0.018373,0.000866,0.000578,0.000098,0.01162,0.007896,0.001007,0.00248,0.003884,0.004348,0.001625,0.000446,0.079863,0.015436,0.007775,0.006056,0.01704,0.024892,0.00637,0.060687,0.021301,0.000363,0.001095,0.364559,0.121748,0.000303,0.283081
1750002,0.011093,0.003459,0.04147,0.013973,0.002312,0.058607,0.009201,0.00173,0.007559,0.003358,0.002907,0.000217,0.000086,0.124412,0.068347,0.00202,0.002231,0.000093,0.002588,0.017173,0.00118,0.003391,0.009178,0.004393,0.001856,0.000446,0.064938,0.017074,0.003271,0.012668,0.081574,0.004649,0.003526,0.212161,0.033495,0.006913,0.020345,0.287256,0.120733,0.000321,0.19569
1750003,0.002013,0.003572,0.018258,0.046158,0.001395,0.012601,0.017999,0.001209,0.003865,0.001606,0.002974,0.000257,0.000086,0.135648,0.028924,0.00095,0.000552,0.000098,0.028622,0.008853,0.000684,0.004939,0.004591,0.003614,0.003854,0.000446,0.016617,0.012764,0.002436,0.021257,0.026045,0.009734,0.007449,0.099205,0.013871,0.000363,0.00052,0.284439,0.086243,0.001374,0.269725
1750004,0.000444,0.00083,0.003874,0.006268,0.000371,0.010673,0.005048,0.00056,0.010326,0.001376,0.001865,0.000257,0.000086,0.119344,0.078902,0.0015,0.000852,0.000093,0.004008,0.015603,0.001123,0.003028,0.005966,0.004448,0.001479,0.000446,0.063364,0.014569,0.006637,0.007252,0.003016,0.01658,0.005463,0.030376,0.030552,0.000363,0.004763,0.34892,0.138135,0.000304,0.223952
1750005,0.008009,0.002908,0.019878,0.014537,0.000381,0.013664,0.011877,0.001516,0.011452,0.003498,0.004492,0.000257,0.000086,0.17463,0.063013,0.001197,0.001417,0.000094,0.000975,0.00622,0.000995,0.007588,0.006435,0.001559,0.002129,0.000446,0.001382,0.009306,0.001848,0.02873,0.021159,0.005161,0.004949,0.074392,0.025235,0.000363,0.010893,0.201438,0.116766,0.00039,0.220593
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999996,0.000348,0.000978,0.002863,0.009907,0.000557,0.000823,0.014216,0.000578,0.002041,0.0009,0.005058,0.000257,0.000048,0.143295,0.059138,0.002222,0.000298,0.000093,0.000523,0.003971,0.002682,0.012617,0.010547,0.002339,0.002686,0.000446,0.029443,0.006013,0.020454,0.017708,0.030297,0.00199,0.000266,0.007833,0.014872,0.00329,0.003354,0.220837,0.012695,0.002097,0.414925
1999997,0.00756,0.002965,0.011771,0.008159,0.001512,0.013796,0.009321,0.004296,0.003998,0.00139,0.003319,0.000256,0.000076,0.184567,0.454107,0.001033,0.012159,0.000167,0.009688,0.020103,0.000732,0.004813,0.006754,0.005504,0.005058,0.000494,0.118747,0.013568,0.001823,0.039923,0.019918,0.004484,0.004083,0.046369,0.008919,0.000363,0.001297,0.122917,0.112096,0.017857,0.210902
1999998,0.000116,0.00053,0.000747,0.000507,0.00031,0.002459,0.00068,0.001581,0.001865,0.000841,0.004175,0.000257,0.000067,0.128462,0.423604,0.000624,0.003446,0.01951,0.001213,0.013654,0.000989,0.017949,0.002079,0.003405,0.025275,0.000504,0.015314,0.000648,0.001819,0.806726,0.006445,0.009147,0.007146,0.010771,0.005804,0.000363,0.000467,0.070886,0.029689,0.002375,0.0409


In [ ]:
'''
Stacking
'''
from sklearn.ensemble import StackingClassifier
from sklearn.multioutput import MultiOutputClassifier

level1_estimators = [
    ('hist_grad', HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.03,
        random_state=42,
        verbose=0
    )),
    ('log_reg', LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ))
]

# Мета-модель
meta_learner = CatBoostClassifier(
    iterations=100,
    depth=4,
    learning_rate=0.03,
    l2_leaf_reg=10,
    task_type='GPU',
    verbose=False,
    random_state=42
)

# StackingClassifier
stacking_clf = StackingClassifier(
    estimators=level1_estimators,
    final_estimator=meta_learner,
    cv=3,  # 3-fold кросс-валидация
    stack_method='predict_proba',
    n_jobs=-1,
    verbose=1
)

# Оборачиваем в MultiOutputClassifier для 41 таргета
super_model = MultiOutputClassifier(stacking_clf, n_jobs=-1)

